# Bandlet transform on TPU: a static-shape decode

The GPU version's decode gathers only the leaves actually chosen by the
quadtree RD decision, batched by leaf count -- fast on GPU (5 rounds of
profiling got it there), but the leaf count is DATA-DEPENDENT, so every
image produces a different tensor shape. TPUs compile a static XLA graph
that wants fixed shapes; a shape that changes per image forces a recompile
per image, which would be far slower than even an unoptimized GPU decode.

This notebook redesigns decode around a mask-select pattern instead of
gather-by-dynamic-index: compute the inverse for EVERY candidate block at
EVERY quadtree level unconditionally (shapes fixed by image size alone),
then use a boolean mask to pick, per pixel, which level's reconstruction is
the real one. Strictly more FLOPs than the GPU version, all of them
static-shape and embarrassingly parallel -- trading compute for
compileability.

**What's actually verified here:** exact correctness of the static decode,
confirmed against the dynamic (GPU-style) decode it's meant to replace, on
every device this was tested on. **What's NOT verified:** actual XLA
compilation behavior and TPU speed -- I have no TPU to test against. The
timing cell runs both decode paths side by side specifically so a real TPU
run produces real evidence rather than asking you to trust a prediction.

**Before running:** `Runtime -> Change runtime type -> TPU`. Everything
also runs correctly on CPU/GPU runtimes (useful for a first correctness
pass before spending TPU time), but the interesting comparison only shows
up on TPU.


## 0. Setup and device detection

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from skimage import data as skdata

torch.set_grad_enabled(False)

# ----------------------------------------------------------------------------
# TPU detection. torch_xla is Google's PyTorch/XLA bridge -- it's what lets
# PyTorch code run on TPUs. It compiles a static computation graph via XLA,
# which is the whole reason this notebook's decode is structured differently
# from the GPU version: XLA wants FIXED tensor shapes known at compile time,
# and the GPU-optimized decode (gather only the leaves that were actually
# chosen, batch by leaf count) produces a DIFFERENT shape for every image,
# forcing a recompile every single time -- which would be far slower than
# even the un-optimized "naive" GPU decode from early rounds of that work.
# ----------------------------------------------------------------------------
HAS_TPU = False
try:
    import torch_xla.core.xla_model as xm
    DEVICE_TPU = xm.xla_device()
    HAS_TPU = True
except ImportError:
    DEVICE_TPU = None

DEVICE_CPU = torch.device('cpu')
HAS_CUDA = torch.cuda.is_available()
DEVICE_GPU = torch.device('cuda') if HAS_CUDA else None

print(f"PyTorch version : {torch.__version__}")
print(f"TPU available   : {HAS_TPU}" + (f"  ({DEVICE_TPU})" if HAS_TPU else ""))
print(f"CUDA available  : {HAS_CUDA}" + (f"  ({torch.cuda.get_device_name(0)})" if HAS_CUDA else ""))
if not HAS_TPU:
    print("\nNo TPU detected. In Colab: Runtime -> Change runtime type -> TPU.")
    print("Everything here still runs correctly on CPU/GPU -- useful for checking")
    print("correctness before spending TPU time, but the whole point of the static")
    print("decode is TPU compilation behavior, which only a real TPU can confirm.")

# Every device this notebook can meaningfully test on, in order.
DEVICES = [DEVICE_CPU] + ([DEVICE_GPU] if HAS_CUDA else []) + ([DEVICE_TPU] if HAS_TPU else [])


def sync(device):
    """Force outstanding work on `device` to complete, for accurate timing.
    CUDA: torch.cuda.synchronize(). TPU: xm.mark_step() schedules execution
    of the accumulated lazy graph; following it with a small .cpu() forces
    the result to actually materialize before the timer stops. This
    methodology is my best understanding of current torch_xla idiom, NOT
    verified against real hardware -- if TPU timings look suspiciously flat
    or suspiciously fast, this is the first thing to double check (newer
    torch_xla versions may prefer torch_xla.sync() instead of xm.mark_step())."""
    if getattr(device, 'type', None) == 'cuda':
        torch.cuda.synchronize()
    elif HAS_TPU and device == DEVICE_TPU:
        xm.mark_step()


## 1. Reversible 5/3 lifting (unchanged from the GPU version)

In [ ]:
# ============================================================================
# Reversible 5/3 lifting wavelet transform. Unchanged from the GPU version --
# every op here already has shapes that depend only on the input's own
# shape, never on data values, so this was already XLA-friendly.
# ============================================================================

def lift53_forward(x, dim):
    x = x.transpose(dim, -1)
    s = x[..., 0::2].clone()
    d = x[..., 1::2].clone()
    L = s.shape[-1]
    s_ext = torch.cat([s, s[..., -1:]], dim=-1)
    d = d - 0.5 * (s_ext[..., 0:L] + s_ext[..., 1:L + 1])
    d_ext = torch.cat([d[..., 0:1], d], dim=-1)
    s = s + 0.25 * (d_ext[..., 0:L] + d_ext[..., 1:L + 1])
    return s.transpose(dim, -1), d.transpose(dim, -1)


def lift53_inverse(s, d, dim):
    s = s.transpose(dim, -1)
    d = d.transpose(dim, -1)
    L = s.shape[-1]
    d_ext = torch.cat([d[..., 0:1], d], dim=-1)
    s = s - 0.25 * (d_ext[..., 0:L] + d_ext[..., 1:L + 1])
    s_ext = torch.cat([s, s[..., -1:]], dim=-1)
    d = d + 0.5 * (s_ext[..., 0:L] + s_ext[..., 1:L + 1])
    N = 2 * s.shape[-1]
    x = torch.empty(s.shape[:-1] + (N,), dtype=s.dtype, device=s.device)
    x[..., 0::2] = s
    x[..., 1::2] = d
    return x.transpose(dim, -1)


def dwt2d_level(img):
    s, d = lift53_forward(img, dim=-1)
    LL, LH = lift53_forward(s, dim=-2)
    HL, HH = lift53_forward(d, dim=-2)
    return LL, LH, HL, HH


def idwt2d_level(LL, LH, HL, HH):
    s = lift53_inverse(LL, LH, dim=-2)
    d = lift53_inverse(HL, HH, dim=-2)
    return lift53_inverse(s, d, dim=-1)


# ---- self-test: perfect reconstruction on every device ----
print(f"{'device':8s} {'DWT max |err|':>16s}")
for dev in DEVICES:
    x = torch.randn(64, 256, 256, device=dev)
    LL, LH, HL, HH = dwt2d_level(x)
    r = idwt2d_level(LL, LH, HL, HH)
    sync(dev)
    print(f"{str(dev):8s} {(x - r).abs().max().item():16.3e}")


## 2. Shear + per-leaf directional lifting (unchanged from the GPU version)

In [ ]:
# ============================================================================
# Integer circular shear + per-leaf directional lifting with exact discrete
# direction search. Unchanged from the GPU version -- shapes here depend
# only on the input block's own shape (B, H, W all fixed by the caller),
# never on data values, so this is already XLA-friendly as written.
# ============================================================================

DIRS = [-3, -2, -1, 0, 1, 2, 3]
MIN_BLOCK = 4
SKIP_K = 999.0


def shear_batch(blocks, ks):
    B, H, W = blocks.shape
    device = blocks.device
    rows = torch.arange(H, device=device).view(1, H, 1).float()
    shift = torch.round(ks.view(B, 1, 1) * rows).long().expand(B, H, W)
    col_idx = torch.arange(W, device=device).view(1, 1, W).expand(B, H, W)
    src_idx = torch.remainder(col_idx - shift, W)
    return torch.gather(blocks, 2, src_idx)


def unshear_batch(blocks, ks):
    B, H, W = blocks.shape
    device = blocks.device
    rows = torch.arange(H, device=device).view(1, H, 1).float()
    shift = torch.round(ks.view(B, 1, 1) * rows).long().expand(B, H, W)
    col_idx = torch.arange(W, device=device).view(1, 1, W).expand(B, H, W)
    src_idx = torch.remainder(col_idx + shift, W)
    return torch.gather(blocks, 2, src_idx)


def best_leaf_batched(blocks, dirs=DIRS):
    B, H, W = blocks.shape
    device = blocks.device
    nd = len(dirs)
    ks = torch.tensor(dirs, device=device, dtype=torch.float32)

    blocks_rep = blocks.unsqueeze(1).expand(B, nd, H, W).reshape(B * nd, H, W)
    ks_rep = ks.view(1, nd).expand(B, nd).reshape(B * nd)

    sheared = shear_batch(blocks_rep, ks_rep)
    s, d = lift53_forward(sheared, dim=1)
    cost = s.abs().sum(dim=(1, 2)) + d.abs().sum(dim=(1, 2))
    cost = cost.view(B, nd)

    s_skip = blocks[:, 0::2, :]
    d_skip = blocks[:, 1::2, :]
    cost_skip = s_skip.abs().sum(dim=(1, 2)) + d_skip.abs().sum(dim=(1, 2))

    all_cost = torch.cat([cost, cost_skip.view(B, 1)], dim=1)
    best_idx = torch.argmin(all_cost, dim=1)
    best_cost = all_cost.gather(1, best_idx.view(B, 1)).squeeze(1)
    is_skip = (best_idx == nd)
    safe_idx = torch.where(is_skip, torch.zeros_like(best_idx), best_idx)

    Hh = H // 2
    s = s.view(B, nd, Hh, W)
    d = d.view(B, nd, Hh, W)
    idx = safe_idx.view(B, 1, 1, 1).expand(B, 1, Hh, W)
    s_lifted = s.gather(1, idx).squeeze(1)
    d_lifted = d.gather(1, idx).squeeze(1)

    is_skip3 = is_skip.view(B, 1, 1)
    best_s = torch.where(is_skip3, s_skip, s_lifted)
    best_d = torch.where(is_skip3, d_skip, d_lifted)
    best_k = torch.where(is_skip, torch.full((B,), SKIP_K, device=device), ks[safe_idx])
    return best_cost, best_k, best_s, best_d


def leaf_inverse_batched(s, d, ks, H):
    is_skip = (ks == SKIP_K)
    lifted = lift53_inverse(s, d, dim=1)
    B, Hh, W = s.shape
    raw = torch.empty(B, H, W, dtype=s.dtype, device=s.device)
    raw[:, 0::2, :] = s
    raw[:, 1::2, :] = d
    out = torch.where(is_skip.view(-1, 1, 1), raw, lifted)
    ks_for_unshear = torch.where(is_skip, torch.zeros_like(ks), ks)
    return unshear_batch(out, ks_for_unshear)


# ---- self-test: exactly invertible on every device ----
print(f"{'device':8s} {'leaf max |err|':>16s}")
for dev in DEVICES:
    torch.manual_seed(1)
    blocks = torch.randn(50, 8, 8, device=dev)
    cost, k, s, d = best_leaf_batched(blocks)
    rec = leaf_inverse_batched(s, d, k, 8)
    sync(dev)
    print(f"{str(dev):8s} {(blocks - rec).abs().max().item():16.3e}")


## 3. Encode: quadtree segmentation, without the host syncs

In [ ]:
# ============================================================================
# Level-synchronous RD-optimal quadtree segmentation (encode). This is
# UNCHANGED in spirit from the GPU version's DP, with one deliberate
# omission: no `.cpu().tolist()` calls. `is_leaf` and `k` stay as device
# tensors throughout -- pulling them to host was how the GPU version fed a
# fast Python-side tree walk (fine for GPU: a normal blocking call), but
# under XLA that forces the lazy graph to materialize at that exact point,
# fragmenting one compileable graph into many small ones. Every op below
# has shapes fixed by (N, MIN_BLOCK) alone -- G at each level is a Python
# int known before any tensor is touched, so this traces to one static
# graph regardless of image content.
# ============================================================================

def blocks_from_grid(sub, bsz):
    N = sub.shape[0]
    G = N // bsz
    x = sub.view(G, bsz, G, bsz).permute(0, 2, 1, 3).reshape(G * G, bsz, bsz)
    return x, G


def blocks_to_grid(blocks, G, bsz):
    """Exact inverse of blocks_from_grid -- reassembles a (G*G,bsz,bsz)
    batch of blocks back into one (N,N) canvas. Same permutation used in
    both directions (swapping the same two axes twice is its own inverse)."""
    N = G * bsz
    return blocks.reshape(G, G, bsz, bsz).permute(0, 2, 1, 3).reshape(N, N)


def segment_subband(sub, lam=1.0, dirs=DIRS, min_block=MIN_BLOCK):
    """Bottom-up RD-optimal quadtree over `sub` (N,N). Returns a list of
    per-level dicts; `is_leaf` and `k` are kept as device tensors only."""
    N = sub.shape[0]
    levels = []

    bsz = min_block
    blocks, G = blocks_from_grid(sub, bsz)
    cost, k, s, d = best_leaf_batched(blocks, dirs)
    total_cost = (cost + lam).view(G, G)
    is_leaf = torch.ones(G, G, dtype=torch.bool, device=sub.device)
    levels.append(dict(bsz=bsz, G=G, total_cost=total_cost, is_leaf=is_leaf,
                        k=k.view(G, G), s=s, d=d))

    while bsz < N:
        bsz *= 2
        blocks, G = blocks_from_grid(sub, bsz)
        cost, k, s, d = best_leaf_batched(blocks, dirs)
        leaf_total = (cost + lam).view(G, G)

        prev = levels[-1]
        child_cost = prev['total_cost'].view(G, 2, G, 2).sum(dim=(1, 3))
        leaf_here = leaf_total <= child_cost
        total_cost = torch.where(leaf_here, leaf_total, child_cost)

        levels.append(dict(bsz=bsz, G=G, total_cost=total_cost, is_leaf=leaf_here,
                            k=k.view(G, G), s=s, d=d))
    return levels


def bandlet_forward(img, lam=1.0, dirs=DIRS, min_block=MIN_BLOCK):
    LL, LH, HL, HH = dwt2d_level(img)
    trees = {name: segment_subband(sub, lam, dirs, min_block)
             for name, sub in [('LH', LH), ('HL', HL), ('HH', HH)]}
    return LL, trees


## 4. Decode, path A: dynamic (GPU-style) -- comparison baseline

In [ ]:
# ============================================================================
# DECODE, PATH A: the GPU-optimized "dynamic" decode (rounds 1-5 of the GPU
# work). Included here ONLY as a comparison baseline -- this is exactly the
# kind of decode that should compile badly under XLA, since the number of
# leaves gathered per (subband, block size) group depends on image content,
# giving every image its own tensor shape and forcing recompilation. Kept
# so you can measure that directly on real TPU hardware rather than take it
# on faith -- see section further down for the side-by-side benchmark.
# ============================================================================

def collect_leaves_dynamic(levels):
    """Pulls is_leaf/k to host ONCE PER LEVEL (not per node -- that specific
    fix from GPU round 1 was real, just not the dominant cost there). This
    host sync is fine on GPU; under XLA it forces the lazy graph to
    materialize at this exact point."""
    is_leaf_host = [lvl['is_leaf'].cpu().tolist() for lvl in levels]
    k_host = [lvl['k'].cpu().tolist() for lvl in levels]

    top = len(levels) - 1
    out = []

    def recurse(level_idx, gr, gc):
        lvl = levels[level_idx]
        bsz = lvl['bsz']
        if is_leaf_host[level_idx][gr][gc] or level_idx == 0:
            flat = gr * lvl['G'] + gc
            out.append((level_idx, flat, gr * bsz, gc * bsz, bsz, k_host[level_idx][gr][gc]))
        else:
            recurse(level_idx - 1, 2 * gr, 2 * gc)
            recurse(level_idx - 1, 2 * gr, 2 * gc + 1)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc)
            recurse(level_idx - 1, 2 * gr + 1, 2 * gc + 1)

    recurse(top, 0, 0)
    return out


def bandlet_inverse_dynamic(LL, trees):
    N = LL.shape[-1]
    device = LL.device

    from collections import defaultdict
    by_bsz = defaultdict(list)
    for name in ('LH', 'HL', 'HH'):
        levels = trees[name]
        by_level = defaultdict(list)
        for ref in collect_leaves_dynamic(levels):
            by_level[ref[0]].append(ref)
        for level_idx, refs in by_level.items():
            lvl = levels[level_idx]
            bsz = lvl['bsz']
            flat_idxs = torch.tensor([r[1] for r in refs], device=device, dtype=torch.long)
            s = lvl['s'].index_select(0, flat_idxs)
            d = lvl['d'].index_select(0, flat_idxs)
            ks = torch.tensor([r[5] for r in refs], device=device, dtype=torch.float32)
            r0s = [r[2] for r in refs]
            c0s = [r[3] for r in refs]
            by_bsz[bsz].append((name, s, d, ks, r0s, c0s))

    outs = {name: torch.zeros(N, N, device=device) for name in ('LH', 'HL', 'HH')}
    for bsz, pieces in by_bsz.items():
        names = [n for (n, s, d, ks, r0s, c0s) in pieces for _ in r0s]
        s_all = torch.cat([p[1] for p in pieces], dim=0)
        d_all = torch.cat([p[2] for p in pieces], dim=0)
        ks_all = torch.cat([p[3] for p in pieces], dim=0)
        r0s_all = [r0 for p in pieces for r0 in p[4]]
        c0s_all = [c0 for p in pieces for c0 in p[5]]
        blocks = leaf_inverse_batched(s_all, d_all, ks_all, bsz)

        B = len(r0s_all)
        r0s_t = torch.tensor(r0s_all, device=device)
        c0s_t = torch.tensor(c0s_all, device=device)
        rng = torch.arange(bsz, device=device)
        row_idx = (r0s_t.view(-1, 1, 1) + rng.view(1, bsz, 1)).expand(-1, bsz, bsz)
        col_idx = (c0s_t.view(-1, 1, 1) + rng.view(1, 1, bsz)).expand(-1, bsz, bsz)
        flat_idx = (row_idx * N + col_idx).reshape(B, -1)
        flat_blocks = blocks.reshape(B, -1)

        for name in ('LH', 'HL', 'HH'):
            sel = [i for i, n in enumerate(names) if n == name]
            if not sel:
                continue
            sel_t = torch.tensor(sel, device=device)
            outs[name].view(-1).scatter_(0, flat_idx[sel_t].reshape(-1), flat_blocks[sel_t].reshape(-1))

    return idwt2d_level(LL, outs['LH'], outs['HL'], outs['HH'])


## 5. Decode, path B: static (TPU-style) -- the new design

In [ ]:
# ============================================================================
# DECODE, PATH B: the TPU-friendly "static" decode. Instead of gathering
# only the leaves that were actually chosen (data-dependent shape), this
# computes the inverse transform for EVERY candidate block at EVERY level
# unconditionally -- shapes fixed by (N, MIN_BLOCK) alone, same as encode --
# then uses a mask to select, per pixel, which level's reconstruction to
# keep. Strictly more FLOPs than the dynamic decode (most candidate blocks
# at most levels aren't the ones actually chosen), but every one of those
# FLOPs is static-shape and embarrassingly parallel, which is exactly what
# a TPU's systolic array wants and exactly what XLA needs to compile once
# and reuse.
#
# The mask logic: a pixel belongs to whichever level is the COARSEST one
# where its containing block was chosen as a leaf (finer levels only get
# used where a coarser ancestor decided to split further). Walking from the
# coarsest level down, "claimed" tracks which pixels already belong to an
# already-resolved leaf; each level's own leaves are exactly the
# not-yet-claimed pixels whose block at THIS level is marked is_leaf.
# ============================================================================

def reconstruct_subband_static(levels, N):
    device = levels[0]['s'].device
    recon = torch.zeros(N, N, device=device)
    claimed = torch.zeros(N, N, dtype=torch.bool, device=device)

    for lvl in reversed(levels):   # coarsest (top) first
        bsz, G = lvl['bsz'], lvl['G']
        is_leaf_full = lvl['is_leaf'].view(G, 1, G, 1).expand(G, bsz, G, bsz).reshape(N, N)
        active = is_leaf_full & (~claimed)
        claimed = claimed | active

        blocks = leaf_inverse_batched(lvl['s'], lvl['d'], lvl['k'].reshape(-1), bsz)
        full_level_recon = blocks_to_grid(blocks, G, bsz)
        recon = torch.where(active, full_level_recon, recon)

    return recon


def bandlet_inverse_static(LL, trees):
    N = LL.shape[-1]
    LH = reconstruct_subband_static(trees['LH'], N)
    HL = reconstruct_subband_static(trees['HL'], N)
    HH = reconstruct_subband_static(trees['HH'], N)
    return idwt2d_level(LL, LH, HL, HH)


# ---- self-test: static decode is exact, and matches the dynamic decode
#      it's meant to replace, on every device ----
print(f"{'device':8s} {'static max |err|':>17s} {'static vs dynamic':>18s}")
for dev in DEVICES:
    torch.manual_seed(2)
    img = torch.rand(64, 64, device=dev)
    LL, trees = bandlet_forward(img, lam=0.5)
    rec_static = bandlet_inverse_static(LL, trees)
    rec_dynamic = bandlet_inverse_dynamic(LL, trees)
    sync(dev)
    err = (img - rec_static).abs().max().item()
    diff = (rec_static - rec_dynamic).abs().max().item()
    print(f"{str(dev):8s} {err:17.3e} {diff:18.3e}")


## 6. Timing: static vs. dynamic, across sizes, every available device

In [ ]:
# ============================================================================
# Timing: static vs dynamic decode, across sizes, on every available device.
# This is the cell that actually answers "does the static redesign help on
# TPU" -- everything above only establishes that it's CORRECT, not fast.
#
# Methodology note (unverified on real TPU): each timed call is preceded by
# a warm-up call (to pay for XLA's one-time compilation cost outside the
# timed region) and followed by sync() to force the lazy graph to actually
# execute before the timer stops. If TPU numbers look suspiciously fast
# (i.e. the graph never actually ran), this is the first thing to check.
# ============================================================================

def load_real_image(size=512):
    img = skdata.camera().astype(np.float32) / 255.0
    H, W = img.shape
    top, left = (H - size) // 2, (W - size) // 2
    return img[top:top + size, left:left + size]


SIZES = [64, 128, 256, 512]

results = {}
for dev in DEVICES:
    print(f"\n--- device={dev} ---")
    results[str(dev)] = {}
    for size in SIZES:
        img_np = load_real_image(size) if size <= 512 else None
        if img_np is None:
            torch.manual_seed(0)
            img = torch.rand(size, size, device=dev)
        else:
            img = torch.from_numpy(img_np).to(dev)

        LL, trees = bandlet_forward(img, lam=0.02)
        sync(dev)

        # warm-up (pays for any one-time compilation outside the timed region)
        _ = bandlet_inverse_static(LL, trees)
        _ = bandlet_inverse_dynamic(LL, trees)
        sync(dev)

        t0 = time.perf_counter()
        rec_static = bandlet_inverse_static(LL, trees)
        sync(dev)
        t1 = time.perf_counter()

        rec_dynamic = bandlet_inverse_dynamic(LL, trees)
        sync(dev)
        t2 = time.perf_counter()

        err_static = (img - rec_static).abs().max().item()
        err_dynamic = (img - rec_dynamic).abs().max().item()

        row = dict(static_ms=1000 * (t1 - t0), dynamic_ms=1000 * (t2 - t1),
                   err_static=err_static, err_dynamic=err_dynamic)
        results[str(dev)][size] = row
        print(f"  size={size:5d}  static={row['static_ms']:9.2f}ms  "
              f"dynamic={row['dynamic_ms']:9.2f}ms  "
              f"err_static={err_static:.2e}  err_dynamic={err_dynamic:.2e}")

print(
"""
On CPU/GPU, dynamic is usually faster or comparable (it's the version that
went through 5 rounds of GPU-specific optimization; static does strictly
more FLOPs by design). The comparison that actually matters is on TPU: if
static wins there -- or if dynamic is dramatically slower / triggers visible
per-call recompilation -- that confirms the shape-based diagnosis. If
they're closer than expected, or static is somehow ALSO slow, that's a real
result worth sending back rather than something to assume away.
""")


## Notes

- **Encode is essentially unchanged.** `segment_subband`'s DP already had
  fixed shapes at every level (G depends only on N and block size, both
  known before any tensor is touched) -- the only edit was removing the
  `.cpu().tolist()` calls that fed the GPU version's host-side tree walk.
  Those calls aren't needed at all for the static decode.
- **Where the redundant compute actually goes:** at every level, EVERY
  candidate block gets inverse-transformed, not just the ones chosen as
  leaves. For a typical image where a modest fraction of candidates end up
  as real leaves, this could be a meaningful multiple of the dynamic
  decode's FLOPs. Whether that's a good trade depends entirely on whether
  TPU's per-op efficiency at fixed shape make up for it -- exactly the
  question the timing cell is built to answer empirically.
- **`xm.mark_step()` vs `torch_xla.sync()`:** the sync() helper in section 0
  uses the older, more universally-recognized `xm.mark_step()` API. Newer
  torch_xla versions introduced `torch_xla.sync()` as the preferred name for
  the same operation. If timing results look off, this is worth checking
  against whatever torch_xla version is actually installed.
- **Recompilation still happens once per distinct image SIZE** (not per
  image) -- that's expected and fine. The problem being solved is
  per-image recompilation from data-dependent leaf counts, not
  recompilation across different sizes, which any static-shape approach
  has to pay for once per size regardless.
- This was developed and correctness-tested against a NumPy-backed shim of
  the torch API (no GPU or TPU in the environment this was written in) --
  exact reconstruction confirmed for the static decode alone, and bit-exact
  agreement with the dynamic decode, across multiple sizes and `lam`
  values including edge cases (`lam=0`: maximal splitting; `lam=1000`:
  full merge to one leaf). Run the self-test cells first when you open this
  on TPU -- if any report a non-negligible error, that's a translation bug
  worth flagging back, not a defect in the design.
